# Train INR Currency Counter on Google Colab (Local Upload)

This notebook trains a **YOLOv8s** (small) model on the **merged 3-dataset** collection — **~5,780 images, 7 classes** (₹5, ₹10, ₹20, ₹50, ₹100, ₹200, ₹500). Expect **~1.5–2 hrs on a T4 GPU**.

### The dataset now includes:
- Original Rajshree dataset + Jay Kumar (front/back) + Project (mccsh) v2
- **Rebalanced** so the previously weak classes are well covered: ₹200 went 91 → 631 boxes, ₹500 went 206 → 801. New-design ₹20 is now covered too.

### Settings:
- **YOLOv8s** (11M params) — best speed/accuracy at this dataset scale
- **Batch 32** on T4 (drop to 16 on OOM)
- **Disk caching** — dataset is now too large to safely cache in Colab RAM
- **AdamW + cosine LR** — fast convergence
- **80 epochs, patience 20** — with early stopping
- Exports both **best.pt** and **best.onnx** for `live_count.py`

## Setup Instructions:
1. On your PC, zip the merged dataset folder: compress `data/merged/` into `merged.zip` (the zip should contain `data.yaml`, `train/`, `valid/` at its top level).
2. Upload `merged.zip` to Colab: click the **Files** icon on the left sidebar and drag the file in.
3. Wait until the **circular upload progress ring** at the bottom-left finishes and disappears.
4. Set runtime to GPU (**Runtime -> Change runtime type -> T4 GPU / L4 GPU**).
5. Run all cells in order.

## Step 1: Install Dependencies
Install the required libraries on the Colab environment:

In [ ]:
!pip install ultralytics torch torchvision opencv-python pyyaml

## Step 2: Verify GPU Acceleration
Verify that Google Colab has assigned a GPU to your runtime session:

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (GPU) available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## Step 3: Unzip Your Uploaded Dataset
Run this cell to extract the uploaded `merged.zip` dataset to the local directory. The `-o` flag ensures any existing files are overwritten, and `-q` runs it silently without showing thousands of files.

In [ ]:
import os

zip_path = '/content/merged.zip'
if os.path.exists(zip_path):
    # Create local data directory
    !mkdir -p /content/data
    
    # Unzip silently and overwrite existing
    !unzip -o -q /content/merged.zip -d /content/data/
    print("Dataset successfully extracted to /content/data!")
    !ls -la /content/data/
else:
    print('Error: Could not find /content/merged.zip.')
    print('Please ensure your merged.zip file has fully finished uploading to the files panel before running this cell.')


## Step 4: Fix and Verify the Dataset Config (data.yaml)
Since the dataset configuration references your local Windows paths, this step automatically rewrites the base path to Colab's Linux directory `/content/data`:

In [ ]:
import yaml

yaml_path = '/content/data/data.yaml'
try:
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)

    # Update base path for Google Colab Linux VM
    cfg['path'] = '/content/data'

    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)
        
    print("data.yaml successfully updated for Colab:")
    with open(yaml_path, 'r') as f:
        print(f.read())
except Exception as e:
    print(f"Error updating data.yaml: {e}")

## Step 5: Download Base Model (YOLOv8s)
Download the pre-trained YOLOv8 Small weights from Ultralytics:

In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt -O /content/yolov8s.pt

## Step 6: Train the YOLOv8s Model
Train your model. This should take **~1.5–2 hours on a T4 GPU** with the larger merged dataset.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s — best speed/accuracy for ~5,780 images, 7 classes
model = YOLO('/content/yolov8s.pt')

# Train with speed-optimized settings
results = model.train(
    data='/content/data/data.yaml',
    epochs=80,             # More data -> allow more epochs; patience stops early if converged
    imgsz=640,
    batch=32,              # YOLOv8s fits batch=32 on T4 (drop to 16 on OOM)
    workers=4,             # Colab has limited CPUs, 8 causes bottlenecks
    project='/content/models',
    name='inr_counter_s',
    cache='disk',          # Dataset now too large for RAM cache on Colab; disk is safe
    optimizer='AdamW',     # Converges much faster than default SGD
    cos_lr=True,           # Cosine LR decay — better late-training convergence
    warmup_epochs=3,
    amp=True,              # Mixed precision — faster on T4 Tensor Cores
    degrees=10.0,          # Random rotation range
    shear=2.0,
    perspective=0.0005,    # Perspective warp for camera angles
    mixup=0.15,            # Mixup augmentation for overlapping notes
    patience=20,           # Stop early if no improvement for 20 epochs
)

## Step 6b: Per-Class Accuracy (check the weak notes)
Confirms how well ₹200 / ₹500 / ₹20 now perform. Look at the **mAP50** column per class — higher is better (>0.8 is good).

In [ ]:
metrics = model.val()
names = model.names
print(f"\nOverall mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}\n")
print(f"{'Denom':<8}{'mAP50':<8}")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"Rs {names[i]:<5}{ap:<8.3f}")

## Step 7: Export to ONNX
Converts `best.pt` to `best.onnx` so `live_count.py` can use the NPU/GPU accelerated path on your PC.

## Step 8: Download Your Trained Weights
Downloads both `best.pt` and `best.onnx` to your PC. Rename them to **`best2.pt`** and **`best2.onnx`** and place in your project's `models/` folder, then run:

```
python live_count.py --model models/best2.onnx --conf 0.4 --verify_time 2 --detect_every 2
```

In [ ]:
from google.colab import files
files.download('/content/models/inr_counter_s/weights/best.pt')
files.download('/content/models/inr_counter_s/weights/best.onnx')

In [ ]:
from google.colab import files
files.download('/content/models/inr_counter_s/weights/best.pt')